# DhakaRoadNet: YOLOv8 Training Pipeline

This notebook starts the training-only stage for **DhakaRoadNet: An Edge AI System for Real-Time Road Object Detection Using a Custom Urban Traffic Dataset**.

The dataset download, verification, class distribution, and visual inspection were completed in `01_dataset_download_and_verification.ipynb`. This notebook focuses on training a pretrained YOLOv8 detector, saving resumable checkpoints, and organizing training outputs for research reporting and later Android/TFLite export.

What this notebook does:
1. Verify the YOLO dataset YAML, class names, and split paths.
2. Explain how YOLOv8 training maps to classical ML and computer vision concepts.
3. Select a suitable pretrained model for an Edge AI baseline.
4. Define a reproducible training configuration.
5. Define realistic urban-road augmentations.
6. Train YOLOv8 with CUDA when available, CPU otherwise.
7. Save weights, checkpoints, metrics, plots, and report-ready artifacts.
8. Provide resume-training and CLI command examples.


## Repository Changes Made for Training

Files used or created by this stage:

- `notebooks/02_training.ipynb`: main training notebook.
- `data/roboflow/data_yolov8.yaml`: local YOLOv8 dataset config generated by notebook 01.
- `model/runs/<experiment_name>/`: Ultralytics training outputs, including `weights/best.pt`, `weights/last.pt`, periodic checkpoints, `results.csv`, `results.png`, and `args.yaml`.
- `model/checkpoints/<experiment_name>/`: copied checkpoint weights for a clean project-level checkpoint location.
- `reports/training/<experiment_name>/`: copied training curves and metrics for research reporting.
- `reports/training_report.md`: experiment report template.

No dataset files should be edited in this notebook.


## Conceptual Bridge: Classical CV/ML to YOLOv8 Training

- **Dataset -> Dataloader -> Batch loading**: YOLO reads image paths and YOLO-format labels from `data_yolov8.yaml`. The dataloader shuffles training samples, applies augmentations, resizes/pads images, stacks them into batches, and feeds tensors to the model.
- **CNN feature extraction**: Convolution layers learn local visual patterns such as road edges, vehicle contours, pothole texture, pedestrian shapes, and lane/scene context.
- **Filters/kernels**: A filter is a small learned weight matrix that slides over the image or feature map. Early filters learn edges/colors; deeper filters learn object parts and object-level patterns.
- **Stride**: Stride controls how far a kernel moves each step. Larger stride reduces spatial resolution and increases receptive field efficiency.
- **Padding**: Padding preserves border information and controls feature-map size after convolution.
- **Downsampling**: YOLO progressively reduces feature-map resolution so deeper layers can understand larger scene context while keeping computation manageable.
- **Backbone**: The backbone extracts multi-scale features from the input image.
- **Neck**: The neck fuses shallow detail and deep semantic features so small objects and large objects can both be detected.
- **Detection head**: The head predicts bounding boxes, object categories, and localization quality at multiple scales.
- **Activation functions**: YOLOv8 commonly uses SiLU-style nonlinear activations in convolution blocks; nonlinear activations let the network model complex visual decision boundaries.
- **Forward propagation**: Images pass through backbone, neck, and head to produce predictions.
- **Loss calculation**: YOLO compares predictions against ground-truth boxes/classes using localization, classification, and box-distribution losses.
- **Backpropagation**: Gradients flow backward from the loss to update filters and model weights.
- **Weight update**: The optimizer applies gradient-based updates to improve predictions over epochs.
- **Optimizer role**: The optimizer controls step size, momentum, and regularization. Here, AdamW is selected for stable transfer learning on a small custom dataset.


## Environment Notes

Your system GPU is an NVIDIA GeForce RTX 3050. The notebook is configured to prefer CUDA device `0` for training.

At inspection time, this Python environment did not have `torch` or `ultralytics` installed, so CUDA training cannot start yet. Install a CUDA-enabled PyTorch build first, then install Ultralytics.

Recommended Windows commands for your current NVIDIA driver/CUDA stack:

```powershell
python -m pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu126
python -m pip install -U ultralytics pandas pyyaml matplotlib opencv-python
```

If PyTorch's selector recommends a newer CUDA wheel for your installed driver, use the selector result from https://pytorch.org/get-started/locally/.

Verify CUDA after installation:

```python
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
```

Important: YOLO/Ultralytics keeps the model on the selected training device and transfers each loaded batch to that same device during training. The dataset files remain on disk; the dataloader reads them through CPU workers and the training loop moves tensors to GPU.


## 1. Imports and Paths

This cell does not import `torch` or `ultralytics` yet. Dataset validation can run even before training dependencies are installed.


In [ ]:
from __future__ import annotations

from pathlib import Path
from pprint import pprint
import csv
import importlib.util
import json
import os
import platform
import random
import shutil
import sys
from datetime import datetime

import matplotlib.pyplot as plt
import pandas as pd
import yaml
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_YAML = PROJECT_ROOT / "data" / "roboflow" / "data_yolov8.yaml"
ORIGINAL_DATA_YAML = PROJECT_ROOT / "data" / "roboflow" / "data.yaml"
MODEL_DIR = PROJECT_ROOT / "model"
RUNS_DIR = MODEL_DIR / "runs"
CHECKPOINT_ROOT = MODEL_DIR / "checkpoints"
REPORTS_DIR = PROJECT_ROOT / "reports"
TRAINING_REPORTS_DIR = REPORTS_DIR / "training"

for directory in [RUNS_DIR, CHECKPOINT_ROOT, TRAINING_REPORTS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print(f"Project root      : {PROJECT_ROOT}")
print(f"YOLO data YAML    : {DATA_YAML}")
print(f"Runs directory    : {RUNS_DIR}")
print(f"Checkpoint root   : {CHECKPOINT_ROOT}")
print(f"Training reports  : {TRAINING_REPORTS_DIR}")
print(f"Python            : {sys.version.split()[0]}")
print(f"Platform          : {platform.platform()}")


## 2. Dependency Check

Training needs PyTorch and Ultralytics. This check is intentionally separate from dataset validation so you can still inspect the data configuration before installing deep-learning packages.


In [ ]:
def package_available(package_name: str) -> bool:
    return importlib.util.find_spec(package_name) is not None


dependency_status = pd.DataFrame(
    [
        {"package": "torch", "installed": package_available("torch")},
        {"package": "ultralytics", "installed": package_available("ultralytics")},
        {"package": "yaml", "installed": package_available("yaml")},
        {"package": "pandas", "installed": package_available("pandas")},
        {"package": "matplotlib", "installed": package_available("matplotlib")},
    ]
)
display(dependency_status)

if not dependency_status.query("package in ['torch', 'ultralytics'] and installed == False").empty:
    print("Install missing training dependencies before running the training cell.")
else:
    print("Training dependencies are installed.")


## 3. Verify Dataset YAML Structure

The training notebook uses `data/roboflow/data_yolov8.yaml`, not the original Roboflow `data.yaml`, because the original file contains split paths that resolve outside this repository layout.


In [ ]:
def load_yaml(path: Path) -> dict:
    if not path.exists():
        raise FileNotFoundError(f"Missing YAML file: {path}")
    with path.open("r", encoding="utf-8") as f:
        return yaml.safe_load(f)


def normalize_names(names) -> list[str]:
    if isinstance(names, dict):
        return [names[k] for k in sorted(names)]
    return list(names)


def resolve_split_path(config: dict, yaml_path: Path, split_key: str) -> Path:
    dataset_root = Path(config.get("path", yaml_path.parent))
    if not dataset_root.is_absolute():
        dataset_root = (yaml_path.parent / dataset_root).resolve()

    split_value = Path(config[split_key])
    if split_value.is_absolute():
        return split_value
    return (dataset_root / split_value).resolve()


def verify_dataset_yaml(yaml_path: Path) -> tuple[dict, list[str], pd.DataFrame]:
    config = load_yaml(yaml_path)
    required_keys = ["path", "train", "val", "test", "nc", "names"]
    missing_keys = [key for key in required_keys if key not in config]
    if missing_keys:
        raise ValueError(f"Dataset YAML is missing required keys: {missing_keys}")

    class_names = normalize_names(config["names"])
    if int(config["nc"]) != len(class_names):
        raise ValueError(f"nc={config['nc']} but names contains {len(class_names)} classes")

    rows = []
    split_map = {"train": "train", "val": "valid", "test": "test"}
    for yaml_key, split_name in split_map.items():
        image_dir = resolve_split_path(config, yaml_path, yaml_key)
        label_dir = image_dir.parent / "labels"
        image_count = sum(1 for p in image_dir.glob("*.*")) if image_dir.exists() else 0
        label_count = sum(1 for p in label_dir.glob("*.txt")) if label_dir.exists() else 0
        rows.append(
            {
                "yaml_key": yaml_key,
                "split": split_name,
                "image_dir": str(image_dir),
                "label_dir": str(label_dir),
                "image_dir_exists": image_dir.exists(),
                "label_dir_exists": label_dir.exists(),
                "images": image_count,
                "labels": label_count,
            }
        )

    split_df = pd.DataFrame(rows)
    if not split_df[["image_dir_exists", "label_dir_exists"]].all().all():
        raise FileNotFoundError("One or more dataset split image/label folders are missing.")

    return config, class_names, split_df


data_config, class_names, split_df = verify_dataset_yaml(DATA_YAML)
print(f"Verified dataset YAML: {DATA_YAML}")
print(f"Number of classes: {len(class_names)}")
display(split_df)
print("Class names:")
for idx, name in enumerate(class_names):
    print(f"  {idx:02d}: {name}")


## 4. Recommended Model Choice

Default model: **`yolov8n.pt`**.

Reasoning:

- This project targets **Edge AI and Android deployment**, so latency, model size, and exportability matter.
- `yolov8n.pt` is the smallest YOLOv8 detection model and is a strong first baseline for TFLite/INT8 export.
- The dataset is modest in size, so transfer learning from pretrained COCO weights is better than training from scratch.
- If the nano model underfits after the baseline, run a second experiment with `yolov8s.pt` and compare mAP vs Android latency.

Do not switch to newer model families in this notebook unless you intentionally change the research baseline away from YOLOv8.


In [ ]:
MODEL_NAME = "yolov8n.pt"
SECONDARY_MODEL_TO_COMPARE_LATER = "yolov8s.pt"

print(f"Primary training baseline : {MODEL_NAME}")
print(f"Later comparison candidate: {SECONDARY_MODEL_TO_COMPARE_LATER}")


## 5. Reproducibility and Device Selection

The notebook uses CUDA when available and CPU otherwise. Deterministic settings improve repeatability, but exact reproducibility can still vary by GPU, driver, CUDA version, and some low-level kernels.


In [ ]:
SEED = 42
random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

# GPU policy for this project.
# RTX 3050 should be CUDA device 0 when PyTorch with CUDA is installed.
USE_GPU = True
GPU_DEVICE_INDEX = 0
ALLOW_CPU_FALLBACK = False


def get_torch_cuda_report() -> dict:
    """Return CUDA/PyTorch status without crashing when torch is missing."""
    report = {
        "torch_installed": package_available("torch"),
        "cuda_available": False,
        "device_count": 0,
        "selected_device": None,
        "selected_device_name": None,
        "torch_version": None,
        "torch_cuda_version": None,
    }
    if not report["torch_installed"]:
        return report

    import torch

    report["torch_version"] = torch.__version__
    report["torch_cuda_version"] = torch.version.cuda
    report["cuda_available"] = torch.cuda.is_available()
    report["device_count"] = torch.cuda.device_count() if report["cuda_available"] else 0
    if report["cuda_available"] and report["device_count"] > GPU_DEVICE_INDEX:
        report["selected_device"] = GPU_DEVICE_INDEX
        report["selected_device_name"] = torch.cuda.get_device_name(GPU_DEVICE_INDEX)
    return report


def select_training_device() -> str | int:
    """Return an Ultralytics-compatible device argument."""
    if USE_GPU:
        return GPU_DEVICE_INDEX
    return "cpu"


def validate_training_device(device: str | int, allow_cpu_fallback: bool = False) -> dict:
    """Fail early if GPU training was requested but CUDA is unavailable."""
    report = get_torch_cuda_report()

    if device == "cpu":
        if not allow_cpu_fallback and USE_GPU:
            raise RuntimeError("CPU fallback is disabled. Install CUDA-enabled PyTorch or set ALLOW_CPU_FALLBACK=True.")
        return {**report, "training_device": "cpu"}

    if not report["torch_installed"]:
        raise ImportError(
            "PyTorch is not installed. Install CUDA-enabled PyTorch before training on the RTX 3050."
        )
    if not report["cuda_available"]:
        if allow_cpu_fallback:
            print("CUDA is not available. Falling back to CPU because ALLOW_CPU_FALLBACK=True.")
            return {**report, "training_device": "cpu"}
        raise RuntimeError(
            "CUDA is not available in PyTorch. Your RTX 3050 is visible to Windows, but this Python environment needs a CUDA-enabled PyTorch install."
        )
    if report["device_count"] <= int(device):
        raise RuntimeError(f"Requested CUDA device {device}, but PyTorch sees only {report['device_count']} CUDA device(s).")

    import torch

    torch.cuda.set_device(int(device))
    return {**report, "training_device": f"cuda:{device}"}


DEVICE = select_training_device()
cuda_report = get_torch_cuda_report()

print(f"USE_GPU            : {USE_GPU}")
print(f"Selected device arg: {DEVICE}")
print(f"Torch installed    : {cuda_report['torch_installed']}")
print(f"CUDA available     : {cuda_report['cuda_available']}")
print(f"CUDA device count  : {cuda_report['device_count']}")
print(f"Selected GPU       : {cuda_report['selected_device_name']}")
print(f"Torch version      : {cuda_report['torch_version']}")
print(f"Torch CUDA version : {cuda_report['torch_cuda_version']}")


## 6. Training Configuration

These values are strong but realistic for a small custom urban-road dataset and an Edge AI baseline.

- `epochs=120`: enough for transfer learning without assuming a huge dataset.
- `patience=25`: early stopping if validation performance plateaus.
- `imgsz=640`: standard YOLO input size and good balance for road objects.
- `batch=16`: good GPU baseline; reduce to 8 or 4 if GPU memory is limited. CPU training will be very slow.
- `optimizer=AdamW`: stable for transfer learning on a smaller dataset.
- `lr0=0.001`: conservative AdamW learning rate.
- `save_period=10`: saves periodic checkpoints so training can resume without starting from the beginning.
- `workers=0`: safer for Windows/Jupyter; increase to 4 or 8 on Linux/cloud training.


In [ ]:
EXPERIMENT_NAME = "yolov8n_dhakaroadnet_baseline"

TRAINING_CONFIG = {
    "model": MODEL_NAME,
    "data": str(DATA_YAML),
    "epochs": 120,
    "patience": 25,
    "batch": 16,  # RTX 3050 6GB: reduce to 8 or 4 if CUDA out-of-memory occurs.
    "imgsz": 640,
    "optimizer": "AdamW",
    "lr0": 0.001,
    "lrf": 0.01,
    "momentum": 0.937,
    "weight_decay": 0.0005,
    "warmup_epochs": 3.0,
    "cos_lr": True,
    "device": DEVICE,
    "workers": 0,
    "seed": SEED,
    "deterministic": True,
    "save": True,
    "save_period": 10,
    "project": str(RUNS_DIR),
    "name": EXPERIMENT_NAME,
    "exist_ok": False,
    "plots": True,
    "val": True,
    "amp": DEVICE != "cpu",
    "cache": False,
}

pprint(TRAINING_CONFIG)


## 7. Recommended Urban-Road Augmentation Configuration

The goal is to improve generalization without creating physically impossible Dhaka road scenes.

| Augmentation | Final value | Why |
|---|---:|---|
| `fliplr` | `0.5` | Useful because left/right road direction can vary; preserves object realism. |
| `flipud` | `0.0` | Harmful because upside-down roads, pedestrians, vehicles, and potholes are unrealistic. |
| `degrees` | `5.0` | Mild camera tilt is realistic for mobile/vehicle footage; large rotation harms horizon geometry. |
| `translate` | `0.10` | Helps with object position variation and partial crops. |
| `scale` | `0.40` | Simulates different distances while avoiding extreme object-size distortion. |
| `shear` | `2.0` | Small perspective/camera alignment changes; high shear makes vehicles unrealistic. |
| `perspective` | `0.0005` | Very mild perspective shift for road scenes; high values distort boxes. |
| `hsv_h` | `0.015` | Small hue shifts for daylight/camera differences. |
| `hsv_s` | `0.60` | Handles saturation variation from weather, cameras, and lighting. |
| `hsv_v` | `0.40` | Useful for shadow, brightness, and overcast/sunny changes. |
| `mosaic` | `0.80` | Strong for small datasets and small objects, but not always realistic, so close it late. |
| `mixup` | `0.05` | Light regularization; higher values create confusing road-object blends. |
| `copy_paste` | `0.0` | Avoid for this detection dataset unless segmentation masks/object cutouts are curated. |
| `close_mosaic` | `15` | Disables mosaic in final epochs so the model finishes on natural images. |


In [ ]:
AUGMENTATION_CONFIG = {
    "fliplr": 0.5,
    "flipud": 0.0,
    "degrees": 5.0,
    "translate": 0.10,
    "scale": 0.40,
    "shear": 2.0,
    "perspective": 0.0005,
    "hsv_h": 0.015,
    "hsv_s": 0.60,
    "hsv_v": 0.40,
    "mosaic": 0.80,
    "mixup": 0.05,
    "copy_paste": 0.0,
    "close_mosaic": 15,
}

pprint(AUGMENTATION_CONFIG)


## 8. Save Training Configuration Snapshot

This makes each experiment reproducible from the files committed or archived with the project.


In [ ]:
config_snapshot = {
    "created_at": datetime.now().isoformat(timespec="seconds"),
    "experiment_name": EXPERIMENT_NAME,
    "model_choice": MODEL_NAME,
    "secondary_model_to_compare_later": SECONDARY_MODEL_TO_COMPARE_LATER,
    "dataset_yaml": str(DATA_YAML),
    "class_count": len(class_names),
    "class_names": class_names,
    "training_config": TRAINING_CONFIG,
    "augmentation_config": AUGMENTATION_CONFIG,
}

experiment_report_dir = TRAINING_REPORTS_DIR / EXPERIMENT_NAME
experiment_report_dir.mkdir(parents=True, exist_ok=True)
config_snapshot_path = experiment_report_dir / "training_config.yaml"
with config_snapshot_path.open("w", encoding="utf-8") as f:
    yaml.safe_dump(config_snapshot, f, sort_keys=False, allow_unicode=False)

print(f"Saved training configuration snapshot: {config_snapshot_path}")


## 9. Training Command

Equivalent CLI command for the configured experiment:


In [ ]:
def format_cli_value(value):
    if isinstance(value, bool):
        return str(value).lower()
    return str(value)


cli_args = {**TRAINING_CONFIG, **AUGMENTATION_CONFIG}
cli_parts = ["yolo", "detect", "train"]
for key, value in cli_args.items():
    cli_parts.append(f"{key}={format_cli_value(value)}")

training_command = " ".join(cli_parts)
print(training_command)


## 10. Start Training

Run this cell only when `torch` and `ultralytics` are installed. Training can take a long time on CPU. On CUDA, monitor GPU memory; if you hit an out-of-memory error, reduce `batch` to `8` or `4`.

Ultralytics will save:

- `model/runs/<experiment_name>/weights/best.pt`
- `model/runs/<experiment_name>/weights/last.pt`
- periodic checkpoint weights because `save_period=10`
- `results.csv`
- `results.png`
- validation prediction plots and training arguments


In [ ]:
RUN_TRAINING = False  # Change to True when you are ready to train on the RTX 3050.

if RUN_TRAINING:
    if not package_available("torch") or not package_available("ultralytics"):
        raise ImportError("Install torch and ultralytics before training.")

    import torch
    from ultralytics import YOLO

    training_device_report = validate_training_device(
        TRAINING_CONFIG["device"],
        allow_cpu_fallback=ALLOW_CPU_FALLBACK,
    )
    print("Training device report:")
    pprint(training_device_report)

    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)
        print(f"Active CUDA device: cuda:{torch.cuda.current_device()} - {torch.cuda.get_device_name(torch.cuda.current_device())}")

    try:
        torch.use_deterministic_algorithms(True, warn_only=True)
    except Exception as exc:
        print(f"Could not force deterministic algorithms: {exc}")

    # Ultralytics receives device=0 below, so it moves the model and each training batch to cuda:0.
    model = YOLO(MODEL_NAME)
    results = model.train(**{**TRAINING_CONFIG, **AUGMENTATION_CONFIG})
    RUN_DIR = Path(results.save_dir)
    print(f"Training complete. Run directory: {RUN_DIR}")
else:
    print("RUN_TRAINING is False. Review the configuration, then set it to True to start GPU training.")


## 11. Copy Weights and Metrics into Clean Project Folders

Run this after training finishes. It copies the important Ultralytics outputs into stable project folders for GitHub/project documentation. Large `.pt` weights are ignored by `.gitignore`, which is usually correct for GitHub; archive them separately if needed.


In [ ]:
def latest_run_dir(runs_dir: Path, experiment_name: str) -> Path:
    exact = runs_dir / experiment_name
    if exact.exists():
        return exact

    candidates = sorted(
        [p for p in runs_dir.glob(f"{experiment_name}*") if p.is_dir()],
        key=lambda p: p.stat().st_mtime,
        reverse=True,
    )
    if not candidates:
        raise FileNotFoundError(f"No run directory found for {experiment_name} in {runs_dir}")
    return candidates[0]


def copy_if_exists(src: Path, dst: Path) -> bool:
    if not src.exists():
        return False
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, dst)
    return True


POSTPROCESS_TRAINING_OUTPUTS = False  # Change to True after a training run exists.

if POSTPROCESS_TRAINING_OUTPUTS:
    run_dir = latest_run_dir(RUNS_DIR, EXPERIMENT_NAME)
    weights_dir = run_dir / "weights"
    checkpoint_dir = CHECKPOINT_ROOT / EXPERIMENT_NAME
    report_dir = TRAINING_REPORTS_DIR / EXPERIMENT_NAME
    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    report_dir.mkdir(parents=True, exist_ok=True)

    copied_files = []
    for weight_file in sorted(weights_dir.glob("*.pt")):
        destination = checkpoint_dir / weight_file.name
        if copy_if_exists(weight_file, destination):
            copied_files.append(destination)

    for artifact_name in ["results.csv", "results.png", "args.yaml", "confusion_matrix.png", "PR_curve.png", "F1_curve.png"]:
        src = run_dir / artifact_name
        dst = report_dir / artifact_name
        if copy_if_exists(src, dst):
            copied_files.append(dst)

    manifest_path = report_dir / "checkpoint_manifest.csv"
    with manifest_path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["file", "size_mb", "modified_time"])
        for file_path in copied_files:
            writer.writerow([
                str(file_path),
                round(file_path.stat().st_size / (1024 * 1024), 3),
                datetime.fromtimestamp(file_path.stat().st_mtime).isoformat(timespec="seconds"),
            ])

    print(f"Copied {len(copied_files)} artifact(s).")
    print(f"Checkpoint directory: {checkpoint_dir}")
    print(f"Report directory    : {report_dir}")
    print(f"Manifest            : {manifest_path}")
else:
    print("POSTPROCESS_TRAINING_OUTPUTS is False. Set it to True after training completes.")


## 12. Resume Training from a Previous Checkpoint

Use this if training stops unexpectedly or if you want to continue from `last.pt`. Ultralytics restores model weights, optimizer state, scheduler state, and epoch progress when `resume=True` is used with `last.pt`.


In [ ]:
RUN_RESUME = False
RESUME_CHECKPOINT = CHECKPOINT_ROOT / EXPERIMENT_NAME / "last.pt"

if RUN_RESUME:
    if not RESUME_CHECKPOINT.exists():
        raise FileNotFoundError(f"Resume checkpoint not found: {RESUME_CHECKPOINT}")
    if not package_available("ultralytics"):
        raise ImportError("Install ultralytics before resuming training.")

    from ultralytics import YOLO

    model = YOLO(str(RESUME_CHECKPOINT))
    resume_results = model.train(resume=True)
    print(f"Resume complete. Run directory: {resume_results.save_dir}")
else:
    print(f"Resume is disabled. Checkpoint path reserved for later: {RESUME_CHECKPOINT}")


## 13. Inspect Learning Curves After Training

Run this after training. It reads `results.csv` and plots train/validation losses and mAP curves if those columns are available.


In [ ]:
PLOT_TRAINING_RESULTS = False

if PLOT_TRAINING_RESULTS:
    run_dir = latest_run_dir(RUNS_DIR, EXPERIMENT_NAME)
    results_csv = run_dir / "results.csv"
    if not results_csv.exists():
        raise FileNotFoundError(f"Missing training results CSV: {results_csv}")

    results_df = pd.read_csv(results_csv)
    results_df.columns = [col.strip() for col in results_df.columns]
    display(results_df.tail())

    plot_groups = {
        "losses": [col for col in results_df.columns if "loss" in col.lower()],
        "metrics": [col for col in results_df.columns if "map" in col.lower() or "precision" in col.lower() or "recall" in col.lower()],
    }

    for title, columns in plot_groups.items():
        if not columns:
            continue
        ax = results_df.plot(x="epoch", y=columns, figsize=(12, 5), title=f"Training {title}")
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        save_path = TRAINING_REPORTS_DIR / EXPERIMENT_NAME / f"{title}_curves.png"
        plt.savefig(save_path, dpi=180)
        plt.show()
        print(f"Saved plot: {save_path}")
else:
    print("PLOT_TRAINING_RESULTS is False. Enable it after training finishes.")


## 14. Output Folder Structure

Expected structure after training and post-processing:

```text
model/
  runs/
    yolov8n_dhakaroadnet_baseline/
      args.yaml
      results.csv
      results.png
      weights/
        best.pt
        last.pt
        epoch10.pt
        epoch20.pt
        ...
  checkpoints/
    yolov8n_dhakaroadnet_baseline/
      best.pt
      last.pt
      epoch10.pt
      epoch20.pt
      ...
reports/
  training/
    yolov8n_dhakaroadnet_baseline/
      training_config.yaml
      results.csv
      results.png
      checkpoint_manifest.csv
      losses_curves.png
      metrics_curves.png
  training_report.md
```


## 15. Training Report Checklist

After each experiment, update `reports/training_report.md` with:

- model name and pretrained weights
- dataset version and class count
- training config
- augmentation config
- hardware and runtime
- best validation metrics
- failure cases
- next experiment decision
- Android deployment implications
